# 난임 임신성공 예측 — 베이스라인 v2 (연구 가설 피처 통합)

> 리서치 스레드의 검증된 가설을 **실제 베이스라인(v1)에 이식**한 버전. 흐름: **STAGE1 근거 재현 → STAGE2 피처 → STAGE3 multi-seed ablation → STAGE4 채택 게이트**.
>
> **정본 기준:** 피처 코드는 실데이터 테스트를 거친 `research_experiment.py`를 정본으로 그대로 이식. (`.md`와 코드가 다르면 `.py` 우선)
>
> **이식 위치 (정본 확인됨):** `add_research_features`는 `transform` **맨 앞**에서 호출 — 원본 문자열(`특정 시술 유형`·`난자 출처`·`나이`)이 인코딩되기 **전**에 파생해야 하기 때문. 모두 행 단위 → **리크 안전**(train/test 동일 적용).
>
> **통합 피처 4종**
> | 토글 | 피처 | 인코딩하는 신호 |
> |---|---|---|
> | `ADD_H3` | `난자출처_기증`, `유효나이` | 기증사이클이면 기증자 나이, 아니면 본인 나이 (성공 좌우하는 진짜 나이) |
> | `ADD_H5` | `ICSI_남성요인`, `ICSI_무남성요인` | ICSI 효과의 **부호 반전**(남성요인 +0.052 vs 비남성 −0.012) |
> | `ADD_H4` | `배반포` | 5일 이식(0.404) vs 3일(0.259) |
> | `ADD_H1` | `난자_과반응21` | 난자 수 역U, 21+ 하락 |
>
> **의도적으로 제외 (중복 회피 — 다시 넣지 말 것):**
> - **H2 수정률(ratio):** 신호는 있으나 개수 피처와 **중복** → 팀 ablation `ratio_OFF≥baseline`과 일치. `ADD_RATIO_FE=False` 유지.
> - **H6 동결×반응:** 이 데이터에서 신호 불명확 → 보류.
>
> **DI 가드:** `배반포`·`난자_과반응21`은 DI 행 입력이 NaN → 0으로 안전 처리(검증 완료).

## 1. 셋업

In [1]:
import re, os, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
def setup_korean_font():
    try:
        import koreanize_matplotlib; return "koreanize_matplotlib"
    except Exception: pass
    from matplotlib import font_manager
    for cand in ["NanumGothic","Malgun Gothic","AppleGothic","NanumBarunGothic"]:
        if any(cand in f.name for f in font_manager.fontManager.ttflist):
            matplotlib.rcParams["font.family"]=cand; matplotlib.rcParams["axes.unicode_minus"]=False; return cand
    matplotlib.rcParams["axes.unicode_minus"]=False; return None
print("폰트:", setup_korean_font() or "미설정", "| lgbm", lgb.__version__)

폰트: 미설정 | lgbm 4.6.0


## 2. 데이터 로드

In [2]:
csvs=sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)) or sorted(glob.glob("**/*.csv", recursive=True))
def pick(paths,*ks):
    for p in paths:
        if all(k in os.path.basename(p).lower() for k in ks): return p
train=pd.read_csv(pick(csvs,"train") or csvs[0]); test=pd.read_csv(pick(csvs,"test"))
sp=pick(csvs,"submission") or pick(csvs,"sample"); sample_sub=pd.read_csv(sp) if sp else None
print("train", train.shape, "| test", test.shape, "| base rate", round(train["임신 성공 여부"].mean(),4))

train (256351, 69) | test (90067, 68) | base rate 0.2583


## 3. 설정 (CONFIG)

In [3]:
CONFIG=dict(
    # 구조 토글 (팀 ablation으로 확정된 값)
    USE_CLINIC_CODE=True,     # 신호 (빼면 -0.0008)
    ADD_RATIO_FE=False,       # 중복 → 끔(확정)
    HIGH_MISSING="flag", PROC_MULTIHOT=True, DROP_HISTORY_LEAK=False,
    # 연구 피처 토글 (STAGE3 ablation으로 채택 결정)
    ADD_H3=False, ADD_H5=False, ADD_H4=False, ADD_H1=False,
    N_SPLITS=5, SEED=42,
)
CONFIG

{'USE_CLINIC_CODE': True,
 'ADD_RATIO_FE': False,
 'HIGH_MISSING': 'flag',
 'PROC_MULTIHOT': True,
 'DROP_HISTORY_LEAK': False,
 'ADD_H3': False,
 'ADD_H5': False,
 'ADD_H4': False,
 'ADD_H1': False,
 'N_SPLITS': 5,
 'SEED': 42}

## 4. 컬럼/매핑 정의

In [4]:
TARGET="임신 성공 여부"; ID_COL="ID"; CLINIC_COL="시술 시기 코드"
COL_PROC="특정 시술 유형"; COL_RSN="배아 생성 주요 이유"
ORDINAL_COUNT_COLS=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수",
 "총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
NOMINAL_COLS=["시술 시기 코드","시술 유형","배란 유도 유형","난자 출처","정자 출처"]
HISTORY_LEAK_SUSPECTS=["총 임신 횟수","총 출산 횟수"]
COUNT_MAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
AGE_T={"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1}
AGE_D={"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}
AGE_MAPS={"시술 당시 나이":AGE_T,"난자 기증자 나이":AGE_D,"정자 기증자 나이":AGE_D}
MALE_COLS=["남성 주 불임 원인","불임 원인 - 남성 요인","불임 원인 - 정자 농도","불임 원인 - 정자 운동성","불임 원인 - 정자 형태"]
print("정의 완료")

정의 완료


## 5. STAGE 1 — 가설 검증 근거 재현

`.py` STAGE1 이식. **Claude의 검증을 '신뢰'가 아니라 '재현'으로 대체**하는 셀. 데이터에서 직접 근거 표를 뽑는다.

In [5]:
def _icsi_male(df):
    icsi=df[COL_PROC].fillna("").str.contains("ICSI")
    male=df[[c for c in MALE_COLS if c in df.columns]].fillna(0).astype(int).max(axis=1).astype(bool)
    return icsi, male

T=TARGET; base=train[T].mean()
print(f"base rate {base:.4f}\n")

# H3
own=train[train["난자 출처"]=="본인 제공"]; don=train[train["난자 출처"]=="기증 제공"]
def slope(s): return s[s["시술 당시 나이"]=="만43-44세"][T].mean()-s[s["시술 당시 나이"]=="만18-34세"][T].mean()
print(f"[H3] 본인 나이기울기 {slope(own):+.3f} vs 기증 {slope(don):+.3f} | "
      f"기증사이클 기증자나이 결측 {don['난자 기증자 나이'].isna().mean():.1%}")
# H5
icsi,male=_icsi_male(train)
g=train.assign(I=icsi,M=male)
e_no=g[~g.M & g.I][T].mean()-g[~g.M & ~g.I][T].mean()
e_ye=g[g.M & g.I][T].mean()-g[g.M & ~g.I][T].mean()
print(f"[H5] 비남성요인 ICSI {e_no:+.3f} / 남성요인 ICSI {e_ye:+.3f}  (부호반전)")
# H4
gd=train.groupby("배아 이식 경과일")[T].mean()
print(f"[H4] 5일 {gd.get(5,float('nan')):.3f} vs 3일 {gd.get(3,float('nan')):.3f}  gap {gd.get(5,0)-gd.get(3,0):+.3f}")
# H1
nb=pd.cut(train["수집된 신선 난자 수"],[-1,3,9,15,20,1000],labels=["1-3","4-9","10-15","16-20","21+"])
gg=train.groupby(nb)[T].mean()
print(f"[H1] 난자수 밴드별 성공률: {gg.round(3).to_dict()}  | 21+ 하락 {gg.get('21+',0)-gg.get('16-20',0):+.3f}")
# H8 누수 재확인
for c in HISTORY_LEAK_SUSPECTS:
    print(f"[H8] {c} 횟수별: {train.groupby(c)[T].mean().round(3).to_dict()}")

base rate 0.2583

[H3] 본인 나이기울기 -0.262 vs 기증 +0.034 | 기증사이클 기증자나이 결측 0.0%
[H5] 비남성요인 ICSI -0.012 / 남성요인 ICSI +0.052  (부호반전)
[H4] 5일 0.404 vs 3일 0.259  gap +0.146
[H1] 난자수 밴드별 성공률: {'1-3': 0.203, '4-9': 0.24, '10-15': 0.324, '16-20': 0.34, '21+': 0.264}  | 21+ 하락 -0.076
[H8] 총 임신 횟수 횟수별: {'0회': 0.257, '1회': 0.263, '2회': 0.258, '3회': 0.245, '4회': 0.215, '5회': 0.25, '6회 이상': 0.333}
[H8] 총 출산 횟수 횟수별: {'0회': 0.257, '1회': 0.267, '2회': 0.263, '3회': 0.278, '4회': 0.231, '5회': 0.0, '6회 이상': 0.0}


## 6. STAGE 2 — 연구 피처 (`.py` 정본 그대로)

In [6]:
def add_research_features(df, cfg):
    df=df.copy()
    if cfg.get("ADD_H5"):
        icsi, male=_icsi_male(df)
        df["ICSI_남성요인"]=(icsi & male).astype(int)
        df["ICSI_무남성요인"]=(icsi & ~male).astype(int)
    if cfg.get("ADD_H3"):
        df["난자출처_기증"]=(df["난자 출처"]=="기증 제공").astype(int)
        df["유효나이"]=np.where(df["난자출처_기증"]==1, df["난자 기증자 나이"].map(AGE_D),
                                 df["시술 당시 나이"].map(AGE_T))
    if cfg.get("ADD_H4"):
        df["배반포"]=(df["배아 이식 경과일"]>=5).astype(int)
    if cfg.get("ADD_H1"):
        df["난자_과반응21"]=(df["수집된 신선 난자 수"]>20).astype(int)
    return df

RESEARCH_COLS=["ICSI_남성요인","ICSI_무남성요인","난자출처_기증","유효나이","배반포","난자_과반응21"]

## 7. 전처리: `fit_preprocessor`(train만) / `transform`(연구피처 먼저)

In [7]:
def _tok_proc(s):   return [] if pd.isna(s) else [t.strip() for t in re.split(r"[/:]",str(s)) if t.strip()]
def _tok_reason(s): return [] if pd.isna(s) else [t.strip() for t in re.split(r"[,/]",str(s)) if t.strip()]
def _safe_div(num,den): return num/den.replace(0,np.nan)

def fit_preprocessor(tr, cfg):
    st={}; ignore={TARGET, ID_COL}
    st["dead_cols"]=[c for c in tr.columns if c not in ignore and tr[c].nunique(dropna=True)<=1]
    st["sparse_cols"]=[c for c in tr.columns if c not in ignore and c not in st["dead_cols"] and tr[c].isna().mean()>0.98]
    nominal=[c for c in NOMINAL_COLS if c in tr.columns]
    if not cfg["USE_CLINIC_CODE"]: nominal=[c for c in nominal if c!=CLINIC_COL]
    if COL_PROC in tr.columns and not cfg["PROC_MULTIHOT"]: nominal=nominal+[COL_PROC]
    st["label_cats"]={c: pd.Index(tr[c].astype("category").cat.categories) for c in nominal}
    if COL_PROC in tr.columns and cfg["PROC_MULTIHOT"]:
        st["proc_vocab"]=sorted({t for L in tr[COL_PROC].apply(_tok_proc) for t in L})
    if COL_RSN in tr.columns:
        st["reason_vocab"]=sorted({t for L in tr[COL_RSN].apply(_tok_reason) for t in L})
    return st

def transform(df_raw, st, cfg):
    df=add_research_features(df_raw, cfg)          # ★ 연구 피처 먼저 (원본 문자열 살아있을 때)
    if TARGET in df.columns: df=df.drop(columns=[TARGET])
    if "시술 유형" in df.columns: df["is_DI"]=(df["시술 유형"]=="DI").astype(int)
    if not cfg["USE_CLINIC_CODE"] and CLINIC_COL in df.columns: df=df.drop(columns=[CLINIC_COL])
    df=df.drop(columns=[c for c in st["dead_cols"] if c in df.columns])
    if cfg.get("DROP_HISTORY_LEAK"):
        df=df.drop(columns=[c for c in HISTORY_LEAK_SUSPECTS if c in df.columns])
    sp=[c for c in st["sparse_cols"] if c in df.columns]
    if cfg["HIGH_MISSING"]=="flag":
        for c in sp: df[f"{c}_있음"]=df[c].notna().astype(int)
        df=df.drop(columns=sp)
    elif cfg["HIGH_MISSING"]=="drop":
        df=df.drop(columns=sp)
    for c in ORDINAL_COUNT_COLS:
        if c in df.columns: df[c]=df[c].map(COUNT_MAP)
    for c,m in AGE_MAPS.items():
        if c in df.columns: df[c]=df[c].map(m)
    cat_features=[]
    for c,cats in st["label_cats"].items():
        if c in df.columns:
            df[c]=pd.Categorical(df[c], categories=cats).codes; cat_features.append(c)
    if cfg["PROC_MULTIHOT"] and "proc_vocab" in st and COL_PROC in df.columns:
        ts=df[COL_PROC].apply(_tok_proc)
        for v in st["proc_vocab"]: df[f"proc_{v}"]=ts.apply(lambda L,v=v:int(v in L))
        df=df.drop(columns=[COL_PROC])
    if "reason_vocab" in st and COL_RSN in df.columns:
        ts=df[COL_RSN].apply(_tok_reason)
        for v in st["reason_vocab"]: df[f"이유_{v}"]=ts.apply(lambda L,v=v:int(v in L))
        df=df.drop(columns=[COL_RSN])
    if cfg["ADD_RATIO_FE"]:                          # 기본 False(중복) — 끔
        def col(c): return df[c] if c in df.columns else pd.Series(np.nan, index=df.index)
        df["수정률"]=_safe_div(col("미세주입에서 생성된 배아 수"), col("미세주입된 난자 수"))
    df=df.drop(columns=[c for c in [ID_COL] if c in df.columns])    # ID 먼저
    obj=[c for c in df.columns if df[c].dtype==object]
    if obj: df=df.drop(columns=obj)                  # 안전망(미인코딩 문자열)
    return df, [c for c in cat_features if c in df.columns]

def make_xy(train_raw, test_raw, cfg):
    st=fit_preprocessor(train_raw, cfg)              # train만 fit
    X_tr, cats=transform(train_raw, st, cfg)
    X_te, _  =transform(test_raw,  st, cfg)
    X_te=X_te.reindex(columns=X_tr.columns)
    y=train_raw[TARGET].astype(int).values; ids=test_raw[ID_COL].values
    return X_tr, y, X_te, ids, [c for c in cats if c in X_tr.columns]

## 8. CV / multi-seed

In [8]:
def run_cv(X, y, cats, cfg, X_test=None, verbose=False):
    skf=StratifiedKFold(n_splits=cfg["N_SPLITS"], shuffle=True, random_state=cfg["SEED"])
    oof=np.zeros(len(X)); fi=np.zeros(X.shape[1]); test_pred=np.zeros(len(X_test)) if X_test is not None else None
    params=dict(objective="binary",metric="auc",learning_rate=0.05,num_leaves=63,feature_fraction=0.8,
                bagging_fraction=0.8,bagging_freq=1,min_child_samples=50,verbose=-1,seed=cfg["SEED"])
    for tri,vai in skf.split(X,y):
        dtr=lgb.Dataset(X.iloc[tri],y[tri],categorical_feature=cats or "auto")
        dva=lgb.Dataset(X.iloc[vai],y[vai],reference=dtr)
        m=lgb.train(params,dtr,num_boost_round=3000,valid_sets=[dva],
                    callbacks=[lgb.early_stopping(100,verbose=False),lgb.log_evaluation(0)])
        oof[vai]=m.predict(X.iloc[vai]); fi+=m.feature_importance("gain")/cfg["N_SPLITS"]
        if X_test is not None: test_pred+=m.predict(X_test)/cfg["N_SPLITS"]
    return roc_auc_score(y,oof), pd.Series(fi,index=X.columns).sort_values(ascending=False), oof, test_pred

def run_cv_multiseed(X,y,cats,cfg,seeds=(42,7,2024)):
    a=[run_cv(X,y,cats,dict(cfg,SEED=s))[0] for s in seeds]
    return float(np.mean(a)), float(np.std(a))

## 9. STAGE 3 — 가설별 multi-seed ablation + STAGE 4 채택 게이트

In [9]:
SEEDS=(42,7,2024)   # 무거우면 (42,7), N_SPLITS=3 로 방향만
CFG_TOGGLES={
    "+H3_유효나이": dict(ADD_H3=True),
    "+H5_ICSI상호": dict(ADD_H5=True),
    "+H4_배반포"  : dict(ADD_H4=True),
    "+H1_과반응"  : dict(ADD_H1=True),
}
ablation={"baseline":{}, **CFG_TOGGLES, "+ALL":dict(ADD_H3=True,ADD_H5=True,ADD_H4=True,ADD_H1=True)}
rows=[]
for name, ov in ablation.items():
    cfg=dict(CONFIG, **ov)
    X,y,Xte,ids,cats=make_xy(train,test,cfg)
    m,s=run_cv_multiseed(X,y,cats,cfg,seeds=SEEDS)
    rows.append({"config":name,"AUC_mean":round(m,5),"AUC_std":round(s,5),"n_feats":X.shape[1]})
    print(f"  {name:<12} {m:.5f} ± {s:.5f}  feats={X.shape[1]}")
res=pd.DataFrame(rows)
b=res.loc[res.config=="baseline","AUC_mean"].iloc[0]
res["Δmean"]=(res["AUC_mean"]-b).round(5)
res["판정"]=np.where(res["Δmean"].abs()<res["AUC_std"],"노이즈","유의?")
print("\n=== STAGE3 ablation ===")
print(res.to_string(index=False))

# STAGE4 게이트: 유의 + 양의 Δ 인 단일 피처만 채택
passed=res[(res.config.isin(CFG_TOGGLES))&(res["판정"]=="유의?")&(res["Δmean"]>0)]
ADOPTED={}
for n in passed.config: ADOPTED.update(CFG_TOGGLES[n])
print("\n★ 채택 토글:", ADOPTED or "없음 — 추가 검토 필요")
print("⚠️ 게이트 미통과 피처는 본 학습으로 넘기지 말 것.")

  baseline     0.73965 ± 0.00013  feats=77
  +H3_유효나이     0.73962 ± 0.00017  feats=79
  +H5_ICSI상호   0.73965 ± 0.00022  feats=79
  +H4_배반포      0.73959 ± 0.00023  feats=78
  +H1_과반응      0.73965 ± 0.00014  feats=78
  +ALL         0.73968 ± 0.00012  feats=83

=== STAGE3 ablation ===
    config  AUC_mean  AUC_std  n_feats    Δmean  판정
  baseline   0.73965  0.00013       77  0.00000 노이즈
  +H3_유효나이   0.73962  0.00017       79 -0.00003 노이즈
+H5_ICSI상호   0.73965  0.00022       79  0.00000 노이즈
   +H4_배반포   0.73959  0.00023       78 -0.00006 노이즈
   +H1_과반응   0.73965  0.00014       78  0.00000 노이즈
      +ALL   0.73968  0.00012       83  0.00003 노이즈

★ 채택 토글: 없음 — 추가 검토 필요
⚠️ 게이트 미통과 피처는 본 학습으로 넘기지 말 것.


## 10. 최종 학습 (채택 피처) + 제출

In [10]:
FINAL_CFG=dict(CONFIG, **ADOPTED)
print("최종 CONFIG 연구토글:", {k:FINAL_CFG[k] for k in ['ADD_H3','ADD_H5','ADD_H4','ADD_H1']})
Xtr,y,Xte,ids,cats=make_xy(train,test,FINAL_CFG)
auc,imp,oof,test_pred=run_cv(Xtr,y,cats,FINAL_CFG,X_test=Xte,verbose=False)
print(f"★ 최종 OOF AUC = {auc:.5f} | 피처 {Xtr.shape[1]}")

# 채택된 연구 피처가 중요도 어디에 있나
inX=[c for c in RESEARCH_COLS if c in Xtr.columns]
if inX:
    print("\n연구 피처 중요도 순위:")
    for c in inX: print(f"  {c}: {list(imp.index).index(c)+1}/{len(imp)}")

sub=sample_sub.copy() if sample_sub is not None else pd.DataFrame({ID_COL:ids})
pc=[c for c in sub.columns if c!=ID_COL]; pc=pc[0] if pc else "probability"
pmap=dict(zip(ids,test_pred))
if ID_COL in sub.columns: sub[pc]=sub[ID_COL].map(pmap)
else: sub[ID_COL]=ids; sub[pc]=test_pred
sub.to_csv("submission.csv", index=False)
print("\nsubmission.csv:", sub.shape, "| 결측", int(sub[pc].isna().sum())); sub.head()

최종 CONFIG 연구토글: {'ADD_H3': False, 'ADD_H5': False, 'ADD_H4': False, 'ADD_H1': False}
★ 최종 OOF AUC = 0.73951 | 피처 77

submission.csv: (90067, 2) | 결측 0


,ID,probability
0,TEST_00000,0.002154
1,TEST_00001,0.001712
2,TEST_00002,0.152733
3,TEST_00003,0.101695
4,TEST_00004,0.519375


## 11. 공유 메모

- **흐름:** STAGE1(근거 재현) → STAGE2(피처) → STAGE3(증분 AUC) → STAGE4(게이트). 신뢰가 아니라 **재현**으로 채택.
- **리크 안전:** 모든 인코더는 train fit, 연구 피처는 행 단위. test는 적용만.
- **중복 회피 (다시 넣지 말 것):** H2 수정률(개수 피처와 중복), H6 동결×반응(신호 불명확).
- **판정 읽는 법:** 9번 `판정`이 "유의?"+Δ>0 인 것만 자동 채택(`ADOPTED`). "노이즈"면 신호 크기와 무관하게 모델 증분 없음 → 버림. (특히 H4 배반포·H1 과반응은 `배아 이식 경과일`·난자수가 이미 수치 피처라 증분이 작을 수 있음 — ablation으로 확인하는 이유.)
- **코드리뷰 주의(이월):** `배아 이식 경과일==0`(24,904건)이 '미이식/취소'인지 규정 확인 → 맞으면 별도 플래그. 종단점(타깃=임신 성공 vs 문헌=생아출생) 크기 해석 주의.
- **다음:** 채택 피처 위에서 하이퍼파라미터 튜닝, clinic 타깃 인코딩(fold 내부 fit 필수), CatBoost/XGB 블렌딩.